In [3]:
import csv

input_file = "sugarcane_irrigation_dataset.csv"
output_file = "irrigation_dataset.csv"

# Mapping text → numbers
soil_map = {
    "Black soil": 1,
    "Clay soil": 2,
    "Sandy soil": 3
}

crop_stage_map = {
    "Early": 1,
    "Middle": 2,
    "Late": 3
}

clean_data = []
seen = set()

with open(input_file, "r") as f:
    reader = csv.reader(f)
    header = next(reader)

    for row in reader:

        # Skip empty rows
        if not row or "" in row:
            continue

        # Convert text to numbers
        row[3] = soil_map.get(row[3], 0)
        row[4] = crop_stage_map.get(row[4], 0)

        row_tuple = tuple(row)

        # Remove duplicates
        if row_tuple not in seen:
            seen.add(row_tuple)
            clean_data.append(row)

# Write cleaned dataset
with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "temperature",
        "humidity",
        "rainfall",
        "soil_type",
        "crop_stage",
        "irrigation"
    ])

    writer.writerows(clean_data)

print("Clean dataset ready for ML training")
print("Saved as:", output_file)

Clean dataset ready for ML training
Saved as: irrigation_dataset.csv


In [1]:
import pandas as pd
data = pd.read_csv("irrigation_dataset.csv")
print(data.head())

   temperature  humidity  rainfall  soil_type  crop_stage
0           25        84       175          3           1
1           35        42       200          3           1
2           20        43       151          1           3
3           39        87        27          3           3
4           40        44        13          2           1


In [2]:
# Rule-based synthetic labels
def generate_label(row):
    if row['rainfall'] < 50 and row['soil_type'] in [2,3]:
        return 1  
    else:
        return 0  

# Apply the function to create irrigation column
data['irrigation'] = data.apply(generate_label, axis=1)
data.to_csv("irrigation_dataset_with_labels.csv", index=False)
print("Synthetic labels added!")

Synthetic labels added!


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle

X = data[["temperature", "humidity", "rainfall", "soil_type", "crop_stage"]]
y = data["irrigation"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Model Accuracy:", accuracy)

# Save trained model for Flask app
pickle.dump(model, open("irrigation_model.pkl", "wb"))
print("Model saved as irrigation_model.pkl")

Model Accuracy: 1.0
Model saved as irrigation_model.pkl
